# Nemotron finetuning pipeline

In [1]:
# ── Config ──────────────────────────────────────────────────────────
LORA_RANK = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0.0

MAX_SEQ_LEN = 8192
NUM_STEPS = 300          # Fewer steps for continual learning
BATCH_SIZE = 32
MICRO_BATCH_SIZE = 4
LEARNING_RATE = 5e-5     # Lower LR to preserve existing knowledge
RESET_WEIGHTS = False    # Load pre-trained 0.85 adapter
MOE_TIE_WEIGHTS = False
SHUFFLE_DATASET = True

TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "up_proj", "down_proj", "in_proj", "out_proj",
    "lm_head",
]
# # Data paths (Kaggle)
# TRAIN_CSV_PATH = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"
# ON_POLICY_PATH = "/kaggle/input/datasets/zhendongli923/on-policy-data/on_policy_data.csv"
# BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
# SEED = 42
# PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
# Data paths (Kaggle) — pre-tokenized corpus from base notebook
CORPUS_PATH = "/kaggle/input/datasets/huikang/huikang-nemotron-repository-snapshot/nemotron-master/training/sft/04-08-16-14/tokens"
TRAIN_ORDER_PATH = "/kaggle/input/datasets/huikang/huikang-nemotron-repository-snapshot/nemotron-master/training/sft/04-08-16-14/logprobs/index.jsonl"
TRAIN_CSV_PATH = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"
#PRETRAINED_ADAPTER_PATH = "/kaggle/input/datasets/zhendongli923/old-weight"
PRETRAINED_ADAPTER_PATH = "/kaggle/input/datasets/dgxchen/trained-adapter"
ON_POLICY_PATH = "/kaggle/input/datasets/punyung/on-policy-data/on_policy_data.csv"
BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
SEED = 42

In [2]:
import os

ON_POLICY_PATH = "/kaggle/input/datasets/punyung/on-policy-data/on_policy_data.csv"

print(os.path.exists(ON_POLICY_PATH))

True


In [3]:
import os
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
print(f"Running on Kaggle: {IS_KAGGLE}")

Running on Kaggle: True


In [4]:
if IS_KAGGLE:
    import subprocess
    subprocess.run(
        "pip install -q --no-index --find-links /kaggle/input/datasets/mayukh18/nemotron-packages/packages "
        "unsloth trl peft transformers datasets accelerate bitsandbytes",
        shell=True, check=True,
    )
    subprocess.run(
        "pip install -q /kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
        shell=True, check=True,
    )
    subprocess.run(
        "pip install -q /kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
        shell=True, check=True,
    )
    print('Kaggle deps installed.')
else:
    print('Non-Kaggle: assuming deps pre-installed.')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.9.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.9.0 which is incompatible.


Kaggle deps installed.


In [5]:
if IS_KAGGLE:
    import sys, shutil, stat
    sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')
    ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
    ptxas_dst = '/tmp/ptxas-blackwell'
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        src_bin = os.path.dirname(ptxas_src)
        dst_bin = '/tmp/triton_nvidia_bin'
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst
        import triton.backends.nvidia as nv_backend
        nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst
    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: '12.0'
    print('Triton env fixes applied.')

Triton env fixes applied.


In [6]:
import json
import zipfile

# ── Load pre-tokenized corpus (from base notebook) ─────────────────
examples = []

if IS_KAGGLE:
    # Load problem_ids in training order from logprobs/index.jsonl
    ordered_ids = []
    seen = set()
    with open(TRAIN_ORDER_PATH) as f:
        for line in f:
            rec = json.loads(line)
            if rec.get('epoch', 0) != 0:
                continue
            pid = rec['problem_id']
            if pid in seen:
                continue
            seen.add(pid)
            ordered_ids.append(pid)
    print(f'Loaded {len(ordered_ids)} problem_ids from training order')

    for sid in ordered_ids:
        seg_path = os.path.join(CORPUS_PATH, sid, 'synthetic.json')
        if not os.path.isfile(seg_path):
            continue
        with open(seg_path) as f:
            rec = json.load(f)
        tokens = rec['tokens']
        mask = rec['mask']
        if not tokens:
            continue
        if len(tokens) > MAX_SEQ_LEN:
            tokens = tokens[:MAX_SEQ_LEN]
            mask = mask[:MAX_SEQ_LEN]
        if not any(mask):
            continue
        examples.append({
            'problem_id': sid,
            'tokens': tokens[:-1],
            'targets': tokens[1:],
            'weights': [float(m) for m in mask[1:]],
            'source': 'base_corpus',
        })
    print(f'Base corpus: {len(examples)} examples')
else:
    print('Non-Kaggle: skipping pre-tokenized corpus (not available locally)')

# ── Load + pre-tokenize on-policy data ─────────────────────────────
opd_count = 0
if IS_KAGGLE and os.path.exists(ON_POLICY_PATH):
    import pandas as pd
    from transformers import AutoTokenizer

    opd_df = pd.read_csv(ON_POLICY_PATH)
    if 'verified' in opd_df.columns:
        opd_df = opd_df[opd_df['verified'] == True].reset_index(drop=True)
    print(f'OPD data: {len(opd_df)} verified rows')

    # Load tokenizer for OPD pre-tokenization
    import kagglehub
    _tok_path = kagglehub.model_download('metric/nemotron-3-nano-30b-a3b-bf16/transformers/default')
    opd_tokenizer = AutoTokenizer.from_pretrained(_tok_path, trust_remote_code=True, local_files_only=True)
    if opd_tokenizer.pad_token is None:
        opd_tokenizer.pad_token = opd_tokenizer.eos_token

    for _, row in opd_df.iterrows():
        prompt = str(row['prompt']).strip() + "\nPlease put your final answer inside `\\boxed{}`."
        # Use cot (full reasoning chain) if available, else answer
        if 'cot' in row and pd.notna(row['cot']):
            answer = str(row['cot']).strip()
        else:
            answer = str(row['answer']).strip()

        # Build chat template strings
        messages = [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": answer},
        ]
        prompt_messages = [{"role": "user", "content": prompt}]

        full_text = opd_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False,
        )
        prompt_text = opd_tokenizer.apply_chat_template(
            prompt_messages, tokenize=False, add_generation_prompt=True,
        )

        # Tokenize
        full_enc = opd_tokenizer(
            full_text, truncation=True, max_length=MAX_SEQ_LEN,
            padding=False, return_tensors='pt',
        )
        prompt_enc = opd_tokenizer(
            prompt_text, truncation=True, max_length=MAX_SEQ_LEN,
            add_special_tokens=False, padding=False, return_tensors='pt',
        )

        tokens = full_enc['input_ids'].squeeze(0).tolist()
        prompt_len = prompt_enc['input_ids'].squeeze(0).shape[0]

        # Build mask: 0.0 for prompt tokens, 1.0 for answer tokens
        weights = [0.0] * min(prompt_len, len(tokens)) + [1.0] * max(0, len(tokens) - prompt_len)
        weights = weights[:len(tokens)]

        if len(tokens) < 2 or not any(weights):
            continue

        pid = str(row.get('id', f'opd_{opd_count}'))
        examples.append({
            'problem_id': pid,
            'tokens': tokens[:-1],
            'targets': tokens[1:],
            'weights': weights[1:],
            'source': 'opd',
        })
        opd_count += 1

    print(f'OPD integrated: {opd_count} examples')
else:
    if not IS_KAGGLE:
        print('Non-Kaggle: skipping OPD integration')
    else:
        print(f'OPD path not found: {ON_POLICY_PATH}')

print(f'\nTotal training examples: {len(examples)}')
base_count = sum(1 for e in examples if e['source'] == 'base_corpus')
print(f'Source: base_corpus={base_count}, opd={opd_count}')

Loaded 7830 problem_ids from training order
Base corpus: 7830 examples
OPD data: 8416 verified rows
OPD integrated: 8416 examples

Total training examples: 16246
Source: base_corpus=7830, opd=8416


In [7]:
def run_training(examples):
    """Continue training 0.85 adapter on curated data."""
    import gc
    import math
    import random
    import time

    import torch
    from unsloth import FastLanguageModel
    from cut_cross_entropy import linear_cross_entropy
    from peft import LoraConfig, load_peft_weights
    from peft.tuners.lora import Linear as LoraLinear

    # ── Model path ────────────────────────────────────────────────
    if IS_KAGGLE:
        import kagglehub
        model_path = kagglehub.model_download('metric/nemotron-3-nano-30b-a3b-bf16/transformers/default')
    else:
        model_path = 'unsloth/Nemotron-3-Nano-30B-A3B'
    print(f'Model path: {model_path}')

    # ── GPU check ─────────────────────────────────────────────────
    import causal_conv1d
    import mamba_ssm
    cc = torch.cuda.get_device_capability(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}, sm_{cc[0]*10+cc[1]}')
    print(f'torch={torch.__version__}, cuda={torch.version.cuda}')
    print(f'mamba_ssm={mamba_ssm.__version__}, causal_conv1d={causal_conv1d.__version__}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

    # ── Load model ────────────────────────────────────────────────
    gc.collect()
    torch.cuda.empty_cache()

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_path,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=False,
        load_in_8bit=False,
        full_finetuning=False,
        trust_remote_code=True,
        unsloth_force_compile=True,
        attn_implementation='eager',
        dtype=torch.bfloat16,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print('Model loaded.')

    # ── Apply LoRA ────────────────────────────────────────────────
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        target_modules=TARGET_MODULES,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias='none',
        use_gradient_checkpointing='unsloth',
        random_state=SEED,
    )
    FastLanguageModel.for_training(model)

    # ── Patch Mamba fast path ─────────────────────────────────────
    nemotron_mod = None
    for _name, _m in sys.modules.items():
        if 'modeling_nemotron_h' in _name and hasattr(_m, 'is_fast_path_available'):
            nemotron_mod = _m
            break
    if nemotron_mod is not None:
        nemotron_mod.is_fast_path_available = True
        print('Patched is_fast_path_available = True')

    # ── Manually add lm_head LoRA ────────────────────────────────
    _causal_lm = model
    while hasattr(_causal_lm, 'model'):
        _causal_lm = _causal_lm.model
    _lm_head = _causal_lm.lm_head
    if not isinstance(_lm_head, LoraLinear):
        _cfg = LoraConfig(r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT)
        model.base_model._create_and_replace(
            _cfg, 'default', target=_lm_head, target_name='lm_head', parent=_causal_lm,
        )
        print('Manually added LoRA to lm_head')

    # ── Load pre-trained adapter weights ──────────────────────────
    if not RESET_WEIGHTS and IS_KAGGLE:
        adapter_weights = load_peft_weights(PRETRAINED_ADAPTER_PATH)
        model_sd = model.state_dict()
        new_sd = {}
        loaded = 0
        for ak, av in adapter_weights.items():
            if ak in model_sd:
                new_sd[ak] = av
                loaded += 1
                continue
            ak_with_default = ak.replace('.lora_A.weight', '.lora_A.default.weight').replace('.lora_B.weight', '.lora_B.default.weight')
            if ak_with_default in model_sd:
                new_sd[ak_with_default] = av
                loaded += 1
                continue
            ak_lm = ak.replace('.backbone.lm_head.', '.lm_head.')
            ak_lm_default = ak_lm.replace('.lora_A.weight', '.lora_A.default.weight').replace('.lora_B.weight', '.lora_B.default.weight')
            if ak_lm_default in model_sd:
                new_sd[ak_lm_default] = av
                loaded += 1
                continue
        model.load_state_dict(new_sd, strict=False)
        print(f'Loaded {loaded}/{len(adapter_weights)} pre-trained adapter weights')
    else:
        print('RESET_WEIGHTS=True or non-Kaggle: using fresh LoRA init')

    # ── Cast LoRA params to fp32 ─────────────────────────────────
    for name, param in model.named_parameters():
        if '.lora_' in name:
            param.data = param.data.to(torch.float32)
    print('LoRA params cast to fp32')

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'Model: {trainable:,} trainable / {total:,} total')

    # ── Patch forward with CCE ───────────────────────────────────
    _base = model
    while hasattr(_base, 'model'):
        _base = _base.model

    def _patched_causal_forward(input_ids=None, attention_mask=None, labels=None, **kwargs):
        backbone_out = _base.backbone(
            input_ids=input_ids, attention_mask=attention_mask,
            **{k: v for k, v in kwargs.items() if k in ('position_ids', 'past_key_values', 'use_cache')},
        )
        hidden_states = backbone_out[0]
        lm_head = _base.lm_head
        base_w = lm_head.base_layer.weight
        lora_A = lm_head.lora_A['default'].weight
        lora_B = lm_head.lora_B['default'].weight
        scaling = lm_head.scaling['default']
        lm_weight = base_w + scaling * lora_B @ lora_A
        if labels is not None:
            per_token_ce = linear_cross_entropy(hidden_states, lm_weight, labels, reduction='none')
            loss = per_token_ce.mean()
        else:
            per_token_ce = None
            loss = None
        model._cached_per_token_ce = per_token_ce
        return loss

    _base.forward = _patched_causal_forward
    print('Patched forward with CCE')

    # ── MoE weight tying ─────────────────────────────────────────
    moe_tied_params = []
    if MOE_TIE_WEIGHTS:
        w1_proj_names = ('gate_up_proj', 'up_proj', 'gate_proj', '.w1.')
        w2_proj_names = ('down_proj', '.w2.')
        for name, param in model.named_parameters():
            if not param.requires_grad or '.experts.' not in name or '.lora_' not in name:
                continue
            is_w1 = any(p in name for p in w1_proj_names)
            is_w2 = any(p in name for p in w2_proj_names)
            is_A = '.lora_A.' in name
            is_B = '.lora_B.' in name
            should_tie = (is_w1 and is_A) or (is_w2 and is_B)
            if should_tie and param.dim() >= 2 and param.shape[0] > 1:
                moe_tied_params.append(param)

        def _tie_param_init():
            with torch.no_grad():
                for p in moe_tied_params:
                    mean = p.data.mean(dim=0, keepdim=True)
                    p.data.copy_(mean.expand_as(p.data))

        def _tie_grads():
            with torch.no_grad():
                for p in moe_tied_params:
                    if p.grad is None:
                        continue
                    grad_sum = p.grad.sum(dim=0, keepdim=True)
                    p.grad.copy_(grad_sum.expand_as(p.grad))

        _tie_param_init()
        print(f'MoE weight tying: {len(moe_tied_params)} params')
    else:
        def _tie_grads():
            pass

    # ── Shuffle ───────────────────────────────────────────────────
    if SHUFFLE_DATASET:
        rng = random.Random(SEED)
        rng.shuffle(examples)
        print(f'Shuffled {len(examples)} examples')

    # ── Training loop ─────────────────────────────────────────────
    gc.collect()
    torch.cuda.empty_cache()

    device = next(model.parameters()).device
    optimizer = None
    max_steps = len(examples) // BATCH_SIZE
    num_steps = min(NUM_STEPS, max_steps)
    print(f'Training: {num_steps} steps, batch={BATCH_SIZE}, micro_batch={MICRO_BATCH_SIZE}, lr={LEARNING_RATE}')

    training_log = []
    step = 0

    for batch_start in range(0, len(examples), BATCH_SIZE):
        if step >= num_steps:
            break
        batch = examples[batch_start:batch_start + BATCH_SIZE]
        batch_tokens = [e['tokens'] for e in batch]
        batch_targets = [e['targets'] for e in batch]
        batch_weights = [e['weights'] for e in batch]

        n = len(batch)
        n_accum = math.ceil(n / MICRO_BATCH_SIZE)
        total_loss_sum = 0.0
        total_weight_sum = 0.0

        for mb_start in range(0, n, MICRO_BATCH_SIZE):
            mb_end = min(mb_start + MICRO_BATCH_SIZE, n)
            mb_toks = batch_tokens[mb_start:mb_end]
            mb_tgts = batch_targets[mb_start:mb_end]
            mb_wts = batch_weights[mb_start:mb_end]

            n_micro = len(mb_toks)
            max_len = max(len(t) for t in mb_toks)

            padded_input = torch.zeros(n_micro, max_len, dtype=torch.long, device=device)
            padded_targets = torch.zeros(n_micro, max_len, dtype=torch.long, device=device)
            padded_weights = torch.zeros(n_micro, max_len, dtype=torch.float32, device=device)
            attention_mask = torch.zeros(n_micro, max_len, dtype=torch.long, device=device)
            for i in range(n_micro):
                seq_len = len(mb_toks[i])
                padded_input[i, :seq_len] = torch.tensor(mb_toks[i], dtype=torch.long)
                padded_targets[i, :seq_len] = torch.tensor(mb_tgts[i], dtype=torch.long)
                padded_weights[i, :seq_len] = torch.tensor(mb_wts[i], dtype=torch.float32)
                attention_mask[i, :seq_len] = 1

            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                model(input_ids=padded_input, attention_mask=attention_mask,
                      labels=padded_targets, use_cache=False)
                per_token_ce = model._cached_per_token_ce
                weighted_loss = per_token_ce * padded_weights
                weight_sum_t = padded_weights.sum()
                loss_sum_t = weighted_loss.sum()
                loss = loss_sum_t / weight_sum_t if weight_sum_t > 0 else loss_sum_t * 0.0

            (loss / n_accum).backward()
            total_loss_sum += loss_sum_t.item()
            total_weight_sum += weight_sum_t.item()
            del loss, per_token_ce, weighted_loss

        if optimizer is None:
            optimizer = torch.optim.AdamW(
                [p for p in model.parameters() if p.requires_grad],
                lr=LEARNING_RATE, betas=(0.9, 0.95), eps=1e-8, weight_decay=0.0,
            )
        lr = LEARNING_RATE * (1 - step / num_steps)
        for pg in optimizer.param_groups:
            pg['lr'] = lr
        _tie_grads()
        grad_norm = torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], max_norm=1e9,
        )
        optimizer.step()
        optimizer.zero_grad()
        loss_mean = total_loss_sum / total_weight_sum if total_weight_sum > 0 else 0
        step += 1
        if step % 10 == 0 or step == 1:
            msg = f'  step {step}/{num_steps}: loss={loss_mean:.6f}, grad_norm={grad_norm:.4f}, lr={lr:.2e}'
            print(msg)
        training_log.append({'step': step, 'loss': loss_mean, 'grad_norm': grad_norm.item(), 'lr': lr})

    print(f'\nTraining complete. Peak VRAM: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB')

    # ── Save adapter ──────────────────────────────────────────────
    from safetensors.torch import load_file, save_file

    save_dir = '/kaggle/working' if IS_KAGGLE else './adapter'
    os.makedirs(save_dir, exist_ok=True)
    for _f in os.listdir(save_dir):
        if _f.startswith('adapter'):
            os.remove(os.path.join(save_dir, _f))
    model.save_pretrained(save_dir)

    # Rename lm_head keys
    st_path = os.path.join(save_dir, 'adapter_model.safetensors')
    tensors = load_file(st_path)
    renamed = {
        k.replace('base_model.model.lm_head.', 'base_model.model.backbone.lm_head.'): v
        for k, v in tensors.items()
    }
    save_file(renamed, st_path)
    print(f'Adapter saved to {save_dir}')

    # Clean unsloth cache
    import shutil
    if os.path.isdir('unsloth_compiled_cache'):
        shutil.rmtree('unsloth_compiled_cache')

    return save_dir, training_log

In [8]:
save_dir, training_log = run_training(examples)

/tmp/ipykernel_65/3936067779.py:9: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-05-26 14:30:44.945314: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779805845.108730      65 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779805845.155595      65 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779805845.561922      65 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779805845.561936      65 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779805845.561938      65 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
Model path: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition, sm_120
torch=2.10.0+cu128, cuda=12.8
mamba_ssm=2.3.1, causal_conv1d=1.6.1
VRAM: 102.0 GB
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.3.17: Fast Nemotron_H patching. Transformers: 4.57.6.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading checkpoint shards:   0%|          | 0/13 [00:00<?, ?it/s]

/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1 does not have a padding token! Will use pad_token = <SPECIAL_999>.
Model loaded.
Unsloth: Detected MoE model with num_experts = 128 and target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'up_proj', 'down_proj', 'in_proj', 'out_proj', 'lm_head']. Enabling LoRA on MoE parameters: ['mlp.experts.gate_up_proj', 'mlp.experts.down_proj']
Unsloth: PEFT set target_parameters but found no matching parameters.
This is expected for MoE models - Unsloth handles MoE expert LoRA targeting separately.
Unsloth: Making `model.base_model.model.backbone` require gradients
Patched is_fast_path_available = True
Loaded 12011/12011 pre-trained adapter weights
LoRA params cast to fp32
Model: 888,154,112 trainable / 32,466,091,456 total
Patched forward with CCE
Shuffled 16246 examples
Training: 300 steps, batch=32, micro_batch=4, lr=5e-05
Unsloth: Will smartly offload gradients to save VRAM!
  step 1/300: loss=0.047671, grad_

/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


Adapter saved to /kaggle/working


In [9]:
import json
import shutil
import zipfile

OUTPUT_DIR = '/kaggle/working' if IS_KAGGLE else '.'
SUBMISSION_DIR = os.path.join(OUTPUT_DIR, 'submission_adapter')
os.makedirs(SUBMISSION_DIR, exist_ok=True)

required_files = ['adapter_config.json', 'adapter_model.safetensors']

for fname in required_files:
    src = os.path.join(save_dir, fname)
    dst = os.path.join(SUBMISSION_DIR, fname)
    if not os.path.exists(src):
        raise FileNotFoundError(f'Missing: {src}')
    shutil.copy2(src, dst)
    print(f'  {fname}: {os.path.getsize(dst)/1024/1024:.1f} MB')

# Patch adapter_config.json
config_path = os.path.join(SUBMISSION_DIR, 'adapter_config.json')
with open(config_path) as f:
    cfg = json.load(f)
cfg['base_model_name_or_path'] = BASE_MODEL_NAME
cfg['inference_mode'] = True
cfg['lora_dropout'] = 0.0
with open(config_path, 'w') as f:
    json.dump(cfg, f, indent=2)

# Create submission.zip
zip_path = os.path.join(OUTPUT_DIR, 'submission.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        zf.write(os.path.join(SUBMISSION_DIR, fname), fname)

with zipfile.ZipFile(zip_path) as zf:
    names = zf.namelist()
    print(f'\nZip contents: {names}')
    assert 'adapter_config.json' in names

print(f'\nsubmission.zip: {os.path.getsize(zip_path)/1024/1024:.1f} MB')
print('Done!')

  adapter_config.json: 0.0 MB
  adapter_model.safetensors: 4061.8 MB

Zip contents: ['adapter_config.json', 'adapter_model.safetensors']

submission.zip: 3640.4 MB
Done!


In [10]:
report = {
    'mode': 'continue_training_0.85',
    'lora_rank': LORA_RANK,
    'lora_alpha': LORA_ALPHA,
    'lora_dropout': LORA_DROPOUT,
    'max_seq_len': MAX_SEQ_LEN,
    'num_steps': NUM_STEPS,
    'batch_size': BATCH_SIZE,
    'micro_batch_size': MICRO_BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'reset_weights': RESET_WEIGHTS,
    'total_examples': len(examples),
    'target_modules': TARGET_MODULES,
    'training_log_last_5': training_log[-5:] if training_log else [],
}
report_path = os.path.join(OUTPUT_DIR, 'train_report.json')
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)
print(f'Report saved: {report_path}')

Report saved: /kaggle/working/train_report.json
